# Pattern 02: BM25 (keyword retrieval)

Follows SPEC.md §8's mandatory 8-section template -- this is one of "the 10 patterns."

**This notebook's committed execution uses `RAG_RECIPES_LLM=mock`** for the generation step (see
`.env.example`). Retrieval itself needs no LLM or embedding model at all -- BM25 is a deterministic
lexical algorithm -- so every retrieval number shown below (hit@k, mrr, the failure examples) is
**100% real**, computed against the real pilot corpus and eval set, regardless of the mock LLM
setting. Only the generated answer *text* and the faithfulness/relevance/citation judge scores are
mocked.


## Reproducibility header (SPEC.md §11)

In [1]:
import platform
import sys
import subprocess
import openai
import numpy

print(f"platform: {platform.platform()}")
print(f"python: {sys.version}")
print(f"openai sdk: {openai.__version__}")
print(f"numpy: {numpy.__version__}")

try:
    git_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd="..").decode().strip()
except Exception:
    git_sha = "(not in a git repo checkout)"
print(f"git commit: {git_sha}")


platform: Windows-11-10.0.26200-SP0
python: 3.12.13 (main, Aug  7 2026, 02:26:41) [MSC v.1944 64 bit (AMD64)]
openai sdk: 2.53.0
numpy: 2.5.2


git commit: 0a53e25169d7b633246d89b6ae127e8fb0ed03fd


## Setup (loaded once, used by every section below)

In [2]:
import os
os.environ.setdefault("RAG_RECIPES_LLM", "mock")

from evals.run import load_corpus_by_id, load_qa_set, run_pattern
from recipes.llm import get_llm, MockLLM

corpus_by_id = load_corpus_by_id("../corpus/corpus.jsonl")
qa_set = load_qa_set("../evals/qa_set.jsonl")
llm = get_llm()  # used for generation (recipe_fn's own LLM calls)

# Judging needs its own backend: under a real API key this is the same
# real model, but under mock, `llm`'s canned generation text isn't valid
# JSON, and the judge prompts require JSON output. A separate MockLLM
# here demonstrates a clean, illustrative run instead of every question
# correctly (but noisily) failing to parse -- see evals/judges.py's
# JudgeParseError and evals/run.py's per-question error isolation.
if os.environ.get("RAG_RECIPES_LLM", "openai").lower() == "mock":
    judge_llm = MockLLM(default_response='{"score": 1, "reasoning": "Mock judge: looks fine."}')
else:
    judge_llm = llm


## 1. What this pattern does

BM25 (Best Matching 25) ranks corpus chunks by lexical term-frequency overlap with the query,
using `rank_bm25`'s implementation (`recipes/bm25.py`'s `BM25Index`, built in P1). No embeddings,
no vector search, no API call for retrieval -- it's a classic information-retrieval baseline that
scores a chunk higher the more query terms it contains, weighted by how rare those terms are
across the whole corpus.


## 2. When to use it

- Your queries share exact vocabulary with the source documents (technical terms, proper nouns,
  model names, acronyms)
- You need retrieval that costs nothing per query and needs no API key
- You want a strong, well-understood baseline to compare fancier patterns against
- Your corpus is small-to-medium and mostly single-language


## 3. When NOT to use it

- Queries are paraphrased far from the source vocabulary (BM25 has no notion of synonymy or
  semantic similarity -- see the real failure examples in section 7)
- The corpus has many near-duplicate documents where fine-grained semantic ranking matters more
  than keyword density
- Multi-hop questions requiring synthesis across two documents that don't share much surface
  vocabulary with each other or with the question (see q13 in section 7)


## 4. Implementation

In [3]:
from recipes.bm25 import make_retrieve_and_answer

retrieve_and_answer = make_retrieve_and_answer(corpus_by_id, llm=llm)

# Try it on one question directly.
sample = retrieve_and_answer("What does PCEval stand for?", k=3)
print("retrieved:", sample.retrieved_chunk_ids)
print("answer:", sample.answer)


retrieved: ['arxiv:2601.02404#0', 'arxiv:2601.00138#0', 'arxiv:2601.00138#2']
answer: This is a mock response.


## 5. Run on our eval set

In [4]:
pattern_fn = make_retrieve_and_answer(corpus_by_id, llm=llm)

result = run_pattern(
    recipe_fn=pattern_fn,
    qa_set=qa_set,
    corpus_by_id=corpus_by_id,
    llm=judge_llm,
    pattern_name="02_bm25",
    judges_enabled=True,
)


=== 02_bm25 (n=18) ===
  hit@3: 0.944  [95% CI 0.833, 1.000]
  hit@10: 1.000  [95% CI 1.000, 1.000]
  mrr: 0.857  [95% CI 0.719, 0.972]
  faithfulness: 1.000  [95% CI 1.000, 1.000]
  answer_relevance: 1.000  [95% CI 1.000, 1.000]
  citation_accuracy: 1.000  [95% CI 1.000, 1.000]
  filter_accuracy: 0.000  [95% CI 0.000, 0.000]
  p50_latency_ms: 2.1
  p95_latency_ms: 2.5
  usd_per_query: $0.01061
  eval_usd: $0.1911


## 6. Example query walkthrough

One example per eval-set category, showing retrieved chunks and the (mocked) final answer.

In [5]:
examples = {
    "keyword": "What does PCEval stand for?",
    "paraphrase": "Why do repeated image generations from the same text prompt in diffusion models end up looking so similar to each other?",
    "multi_hop": "The two photonics-AI-systems papers in this corpus each address a different part of the same challenge. What does each one focus on?",
    "filter": "Among the cs.LG papers in this corpus, which one addresses diagnosing a pregnancy complication using deep learning?",
}

for category, question in examples.items():
    result = retrieve_and_answer(question, k=3)
    print(f"--- {category} ---")
    print(f"Q: {question}")
    print(f"Retrieved: {result.retrieved_chunk_ids}")
    print(f"A: {result.answer}")
    print()


--- keyword ---
Q: What does PCEval stand for?
Retrieved: ['arxiv:2601.02404#0', 'arxiv:2601.00138#0', 'arxiv:2601.00138#2']
A: This is a mock response.

--- paraphrase ---
Q: Why do repeated image generations from the same text prompt in diffusion models end up looking so similar to each other?
Retrieved: ['arxiv:2601.00090#0', 'arxiv:2601.00090#1', 'arxiv:2601.00090#2']
A: This is a mock response.

--- multi_hop ---
Q: The two photonics-AI-systems papers in this corpus each address a different part of the same challenge. What does each one focus on?
Retrieved: ['arxiv:2601.00086#0', 'arxiv:2601.00086#1', 'arxiv:2601.00129#2']
A: This is a mock response.

--- filter ---
Q: Among the cs.LG papers in this corpus, which one addresses diagnosing a pregnancy complication using deep learning?
Retrieved: ['arxiv:2601.00907#0', 'arxiv:2601.00907#2', 'arxiv:2601.00116#1']
A: This is a mock response.



## 7. Where this pattern FAILS

Real failures, computed directly against the pilot corpus + `evals/qa_set.jsonl` (retrieval needs
no API calls, so these numbers are genuine, not mocked -- see the reproducibility note at the top
of this notebook).

**Failure 1 -- q13 (multi-hop), hit@3 = 0.0:** *"The two photonics-AI-systems papers in this
corpus each address a different part of the same challenge. What does each one focus on?"* This
question needs **both** `arxiv:2601.00130#0` and `arxiv:2601.00129#0`. BM25's top-10 for this
query only surfaces one of them, and even that one lands dead last at **rank 10** (score 14.3, far
behind the rank-1 chunk's score of 20.6). The question's phrasing ("each address a different part
of the same challenge") is generic English that doesn't share much vocabulary with either paper's
actual technical content, and BM25 has no way to recognize that two *different* documents jointly
answer a *compound* question -- it can only rank single chunks by term overlap with the query as a
whole.

**Failure 2 -- q09 and q12 (paraphrase), correct chunk ranked 2nd, not 1st:** For *"How does the
tool-use adaptation method turn its own past mistakes into guidance for future tasks?"* (q09), the
ground-truth chunk `arxiv:2601.00086#0` (the paper's abstract) scores 12.561 and lands at **rank
2**, narrowly beaten by `arxiv:2601.00086#2` (the paper's methods section, score 12.625) -- the
*same paper*, different section, with marginally higher raw term-frequency density. The identical
pattern shows up in q12 (`arxiv:2601.00121#0` vs `#2`, scores 28.696 vs 29.254). BM25 has no
concept of "this is the canonical/definitional section" -- it purely counts term overlap, so a
methods section that happens to repeat domain vocabulary more densely than the abstract can
outrank the section a human would call the "right" answer, even when both are from the correct
paper.


## 8. Copy-paste snippet

Meant for pasting into your own project, not executed as a cell in this notebook (that's why it's
a markdown code block, not a code cell -- `corpus_by_id = {...}` below is a placeholder for your
own data, not valid standalone Python).

```python
"""Minimal BM25 retrieval + generation, no eval harness."""
from recipes.bm25 import make_retrieve_and_answer
from recipes.llm import get_llm

corpus_by_id = {}  # {chunk_id: {"text": ..., ...}, ...} -- fill in your own chunks
llm = get_llm()

retrieve_and_answer = make_retrieve_and_answer(corpus_by_id, llm=llm)
result = retrieve_and_answer("your question here", k=5)
print(result.answer)
```
